In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import zipfile
import shutil
import random

# 1. 경로 설정
zip_path = '/content/drive/MyDrive/yolo_cls.zip'  # 압축파일 경로
extract_path = '/content/yolo_cls_raw'           # 임시 압축 해제 경로
dataset_path = '/content/car_dataset'           # 최종 YOLO 학습용 경로

# 2. 압축 해제
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 3. 데이터 분할 및 이동
classes = ['front', 'front_left', 'front_right', 'left', 'rear', 'rear_left', 'rear_right', 'right']
split_ratio = 0.8

for cls in classes:
    os.makedirs(os.path.join(dataset_path, 'train', cls), exist_ok=True)
    os.makedirs(os.path.join(dataset_path, 'val', cls), exist_ok=True)

    # 압축 해제된 폴더 내의 클래스 폴더 경로 (구조에 따라 조정 필요할 수 있음)
    src_path = os.path.join(extract_path, cls)
    if not os.path.exists(src_path): # 압축파일 안에 yolo_cls 폴더가 한 겹 더 있을 경우 대비
        src_path = os.path.join(extract_path, 'yolo_cls', cls)

    files = [f for f in os.listdir(src_path) if os.path.isfile(os.path.join(src_path, f))]
    random.shuffle(files)

    split_idx = int(len(files) * split_ratio)
    train_files = files[:split_idx]
    val_files = files[split_idx:]

    for f in train_files:
        shutil.copy(os.path.join(src_path, f), os.path.join(dataset_path, 'train', cls, f))
    for f in val_files:
        shutil.copy(os.path.join(src_path, f), os.path.join(dataset_path, 'val', cls, f))

print(f"데이터셋 준비 완료! 경로: {dataset_path}")

데이터셋 준비 완료! 경로: /content/car_dataset


In [ ]:
!pip install ultralytics

from ultralytics import YOLO

# 모델 로드 (Classification 전용 모델)
model = YOLO('yolov8m-cls.pt')

# 학습 실행
model.train(
    data='/content/car_dataset',
    epochs=50,
    imgsz=224,
    batch=16
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/car_dataset, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freez

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f6e64a01b20>
curves: []
curves_results: []
fitness: 0.9208633303642273
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 0.8489208817481995, 'metrics/accuracy_top5': 0.9928057789802551, 'fitness': 0.9208633303642273}
save_dir: PosixPath('/content/runs/classify/train')
speed: {'preprocess': 0.10744102877710927, 'inference': 3.0300772302175147, 'loss': 0.0005456258984706189, 'postprocess': 0.0007017194266743428}
task: 'classify'
top1: 0.8489208817481995
top5: 0.9928057789802551

In [ ]:
import shutil
from google.colab import files

# 'train' 폴더 전체를 result.zip으로 압축 (경로는 실제 생성된 폴더명에 맞춰 수정하세요)
# 보통 /content/runs/classify/train 폴더에 결과가 저장됩니다.
shutil.make_archive('train_result', 'zip', '/content/runs/classify/train')

# 압축 파일 다운로드
files.download('train_result.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>